In [1]:
import pandas as pd
import sqlite3
import numpy as np

In [2]:
# 1. Load the data from the csv file
df = pd.read_csv("C:/amish/pancrea_proteomics/analysis/data/pancreas_proteomics.csv", header=1)
print("Read file")
print(df.head)

Read file
<bound method NDFrame.head of       Protein ID      Gene Annotated Gene Annotated Matrisome Division  \
0         A6NMZ7    COL6A6         A6NMZ7               Core matrisome   
1         P02452    COL1A1         P02452               Core matrisome   
2         P02458    COL2A1         P02458               Core matrisome   
3         P02461    COL3A1         P02461               Core matrisome   
4         P02462    COL4A1         P02462               Core matrisome   
...          ...       ...            ...                          ...   
7416      Q9Y6X8      ZHX2         Q9Y6X8                Non-matrisome   
7417      Q9Y6X9     MORC2         Q9Y6X9                Non-matrisome   
7418      Q9Y6Y0  IVNS1ABP         Q9Y6Y0                Non-matrisome   
7419      Q9Y6Y8   SEC23IP         Q9Y6Y8                Non-matrisome   
7420  A0A0B4J1V0  IGHV3-15            NaN                          NaN   

     Annotated Matrisome Category  sample_01 MaxLFQ Total Intensity  \


In [3]:
# 2. Extract all annotations from the data
annotations = df[['Protein ID', 'Gene', 'Annotated Gene', 'Annotated Matrisome Division', 'Annotated Matrisome Category']].copy()
#print(annotations)

In [4]:
intensity_cols = [col for col in df.columns if 'MaxLFQ' in col]
expression = df[['Protein ID'] + intensity_cols].copy()
expression_long = expression.melt(id_vars=['Protein ID'], var_name='Sample', value_name='Intensity')
print(expression_long)

        Protein ID                            Sample    Intensity
0           A6NMZ7  sample_01 MaxLFQ Total Intensity    153502.56
1           P02452  sample_01 MaxLFQ Total Intensity  98558584.00
2           P02458  sample_01 MaxLFQ Total Intensity   3466728.80
3           P02461  sample_01 MaxLFQ Total Intensity  70546624.00
4           P02462  sample_01 MaxLFQ Total Intensity  29273744.00
...            ...                               ...          ...
133573      Q9Y6X8  sample_20 MaxLFQ Total Intensity         0.00
133574      Q9Y6X9  sample_20 MaxLFQ Total Intensity     76899.05
133575      Q9Y6Y0  sample_20 MaxLFQ Total Intensity         0.00
133576      Q9Y6Y8  sample_20 MaxLFQ Total Intensity    364424.30
133577  A0A0B4J1V0  sample_20 MaxLFQ Total Intensity         0.00

[133578 rows x 3 columns]


In [5]:
expression_long['Intensity'] = expression_long['Intensity'].replace(0, np.nan)
expression_long['Sample ID'] = expression_long['Sample'].str.extract(r'(sample_\d+)')
expression_long['Condition'] = np.where(expression_long['Sample ID'].isin([f"sample_{i:02d}" for i in [1, 2, 3, 5, 6, 7, 8, 9, 10]]), 'T1D', 'Control')
print(expression_long)

        Protein ID                            Sample    Intensity  Sample ID  \
0           A6NMZ7  sample_01 MaxLFQ Total Intensity    153502.56  sample_01   
1           P02452  sample_01 MaxLFQ Total Intensity  98558584.00  sample_01   
2           P02458  sample_01 MaxLFQ Total Intensity   3466728.80  sample_01   
3           P02461  sample_01 MaxLFQ Total Intensity  70546624.00  sample_01   
4           P02462  sample_01 MaxLFQ Total Intensity  29273744.00  sample_01   
...            ...                               ...          ...        ...   
133573      Q9Y6X8  sample_20 MaxLFQ Total Intensity          NaN  sample_20   
133574      Q9Y6X9  sample_20 MaxLFQ Total Intensity     76899.05  sample_20   
133575      Q9Y6Y0  sample_20 MaxLFQ Total Intensity          NaN  sample_20   
133576      Q9Y6Y8  sample_20 MaxLFQ Total Intensity    364424.30  sample_20   
133577  A0A0B4J1V0  sample_20 MaxLFQ Total Intensity          NaN  sample_20   

       Condition  
0            T1D  
1

In [7]:
conn = sqlite3.connect("../data/pancreas_proteomics.db")
annotations.to_sql('annotations', conn, if_exists='replace', index=False)
expression_long[['Protein ID', 'Sample ID', 'Condition', 'Intensity']].to_sql('expression', conn, if_exists='replace', index=False) 

133578